<a href="https://colab.research.google.com/github/ReemasTariq/NLP/blob/main/Lab6_deep_learning_for_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [2]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [4]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

    df["review"] = df["review"].apply(preprocess_text)
    df.head()

## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [7]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [8]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [9]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [10]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6545
Epoch 10/10 - Loss: 0.4798


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.695


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [25]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.69      0.66      0.67        96
    Positive       0.70      0.73      0.71       104

    accuracy                           0.69       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.69      0.69      0.69       200

Confusion Matrix:
[[63 33]
 [28 76]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [15]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.2 MB/s eta 0:00:00


In [16]:
from gensim.models import Word2Vec
import numpy as np
import torch

# Tokenize the training and testing reviews
X_train_tokens = [review.split() for review in X_train]
X_test_tokens = [review.split() for review in X_test]

# Train Word2Vec using only the training reviews
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1
)

# Represent each review by the average of its word vectors
def review_to_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

X_train_w2v = np.array([
    review_to_vector(review, w2v_model)
    for review in X_train_tokens
])

X_test_w2v = np.array([
    review_to_vector(review, w2v_model)
    for review in X_test_tokens
])

# Convert Word2Vec features and labels to PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)
y_train_w2v_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print("Training shape:", X_train_w2v_tensor.shape)
print("Testing shape:", X_test_w2v_tensor.shape)


Training shape: torch.Size([800, 200])
Testing shape: torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [17]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Input size equals the Word2Vec vector size
input_size = X_train_w2v_tensor.shape[1]

# Neural network: input layer, three hidden layers, and one output neuron
model_w2v = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

# Binary classification loss and Adam optimizer
loss_function_w2v = nn.BCEWithLogitsLoss()
optimizer_w2v = torch.optim.Adam(model_w2v.parameters(), lr=0.001)

# Train for 15 epochs
epochs = 15

for epoch in range(epochs):
    model_w2v.train()

    optimizer_w2v.zero_grad()

    # Forward pass
    outputs = model_w2v(X_train_w2v_tensor)

    # Calculate loss
    loss = loss_function_w2v(outputs, y_train_w2v_tensor)

    # Backpropagation and update weights
    loss.backward()
    optimizer_w2v.step()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

# Evaluate on the test set
model_w2v.eval()

with torch.no_grad():
    test_outputs = model_w2v(X_test_w2v_tensor)
    probabilities = torch.sigmoid(test_outputs)
    predictions_w2v = (probabilities >= 0.5).int().numpy().flatten()

accuracy_w2v = accuracy_score(y_test, predictions_w2v)

print(f"Test Accuracy: {accuracy_w2v:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, predictions_w2v, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions_w2v))


Epoch 1/15 - Loss: 0.6934
Epoch 2/15 - Loss: 0.6933
Epoch 3/15 - Loss: 0.6932
Epoch 4/15 - Loss: 0.6931
Epoch 5/15 - Loss: 0.6931
Epoch 6/15 - Loss: 0.6931
Epoch 7/15 - Loss: 0.6931
Epoch 8/15 - Loss: 0.6931
Epoch 9/15 - Loss: 0.6931
Epoch 10/15 - Loss: 0.6931
Epoch 11/15 - Loss: 0.6931
Epoch 12/15 - Loss: 0.6931
Epoch 13/15 - Loss: 0.6931
Epoch 14/15 - Loss: 0.6931
Epoch 15/15 - Loss: 0.6931
Test Accuracy: 0.480

Classification Report:
              precision    recall  f1-score   support

    Negative       0.48      1.00      0.65        96
    Positive       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
